In [2]:
import numpy as np
import glob
import cv2 as cv

import re

from skimage import morphology, img_as_ubyte

# Fix the issue of Error #15 "Initializing libiomp5md.dll, but found mk2iomp5md.dll already initialized."
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

In [3]:
data_dir = os.path.join('..', 'Data', 'HCT116')
image_dir = os.path.join(data_dir, 'Masks_Binary')
image_paths = glob.glob(os.path.join(image_dir, '*.jpg'))
print(f'Number of Images: {len(image_paths)}')

Number of Images: 787


In [ ]:
ground_truth = [[113, 90, 99, 87, 96, 93, 92, 95, 96, 76, 60, 63, 10, 11, 10, 1, 1, 1],
                [145, 175, 160, 167, 156, 120, 395, 446, 436, 107, 101, 92, 43, 28, 45, 4, 3, 6],
                [103, 112, 72, 74, 76, 100, 62, 74, 79, 71, 83, 79, 8, 11, 10, 3, 2, 12],
                [170, 176, 160, 139, 137, 156, 101, 98, 93, 96, 102, 87, 27, 29, 20, 7, 4, 4],
                [165, 159, 129, 149, 164, 160, 90, 89, 103, 116, 127, 128, 18, 18, 16, 3, 8, 9],#5
                [139, 140, 123, 117, 106, 121, 89, 100, 79, 91, 100, 102, 14, 22, 9, 7, 10, 7],
                [160, 153, 144, 140, 140, 151, 149, 143, 146, 166, 168, 169, 25, 20, 20, 14, 11, 9],
                [157, 160, 117, 138, 140, 155, 144, 118, 117, 138, 125, 108, 27, 29, 25, 6, 11, 10],
                [169, 156, 173, 217, 171, 170, 123, 125, 109, 166, 143, 173, 32, 19, 21, 8, 6, 4],
                [153, 156, 135, 157, 138, 163, 94, 92, 72, 119, 117, 92, 31, 31, 21, 4, 9, 8], #10
                [147, 158, 143, 156, 149, 119, 118, 143, 121, 168, 175, 144, 28, 27, 18, 10, 8, 13],
                [137, 115, 115, 126, 122, 136, 93, 105, 114, 137, 119, 142, 28, 29, 25, 8, 12, 7],
                [228, 240, 294, 259, 243, 239, 1, 1, 0],
                [114, 98, 115, 104, 98, 121, 99, 75, 75, 59, 76, 62, 9, 9, 14, 4, 1, 0],
                [220, 202, 213, 176, 209, 201, 112, 127, 101, 65, 62, 64, 18, 23, 31, 5, 3, 5],
                [145, 152, 167, 164, 167, 166, 114, 82, 74, 59, 12, 10, 13, 1, 3, 4],
                [61, 59, 62, 60, 67, 57, 30, 19, 36, 33, 45, 32, 6, 7, 16, 16, 16, 7, 15, 15, 17, 33, 28, 28],
                [103, 95, 87, 87, 83, 90, 44, 48, 51, 45, 63, 55, 13, 19, 17, 19, 20, 21, 6, 7, 2, 4, 2, 4],
                [74, 72, 93, 65, 76, 87, 60, 55, 49, 59, 49, 66, 16, 11, 9, 15, 12, 14, 4, 2, 5, 7, 4, 8],
                [8, 11, 13, 6, 10, 5, 46, 55, 53, 69, 67, 52, 8, 11, 14, 6, 19, 12, 12, 18, 11, 14, 15, 11],
                [70, 90, 80, 96, 93, 76, 68, 63, 65, 78, 75, 59, 10, 6, 10, 8, 16, 9, 3, 2, 4, 6, 5, 4],
                [43, 48, 46, 34, 41, 41, 12, 16, 8, 25, 24, 20, 3, 7, 12, 6, 7, 5, 4, 2, 2, 1, 2, 0],
                [133, 130, 99, 105, 117, 110, 34, 49, 55, 48, 55, 65, 17, 22, 14, 14, 16, 13, 2, 2, 3, 8, 1, 6],
                [29, 40, 5, 12, 9, 6],
                [118, 102, 108, 101, 112, 113, 46, 45, 35, 47, 54, 45, 11, 16, 16, 12, 20, 17, 3, 2, 1, 4, 3, 1],
                [144, 117, 147, 149, 125, 122, 56, 73, 52, 40, 55, 51, 21, 29, 30, 23, 19, 32, 9, 5, 3, 8, 3, 5],
                [131, 148, 160, 155, 158, 132, 96, 110, 79, 116, 123, 108, 46, 33, 46, 40, 45, 42, 42, 26, 40, 42, 30, 36], #27
                [157, 144, 143, 142, 159, 163, 78, 78, 70, 149, 153, 124, 53, 43, 44, 49, 45, 45, 28, 26, 22, 24, 20, 20],
                [134, 128, 108, 121, 111, 105, 94, 85, 84, 110, 126, 116, 53, 48, 49, 51, 51, 40, 41, 46, 36, 38, 42, 37],
                [159, 134, 163, 165, 179, 147, 94, 95, 92, 158, 149, 158, 41, 39, 39, 38, 45, 47, 36, 30, 32, 34, 36, 25],
                [181, 136, 161, 158, 155, 130, 70, 74, 72, 128, 140, 125, 51, 42, 46, 52, 57, 48, 18, 25, 27, 20, 23, 19],
                [137, 141, 137, 147, 158, 124, 82, 75, 80, 166, 143, 130, 28, 34, 39, 47, 39, 46, 17, 25, 21, 19, 22, 20],
                [146, 146, 164, 125, 150, 160, 111, 110, 98, 134, 120, 129, 64, 46, 52, 60, 43, 55, 36, 34, 42, 31, 28, 36],
                [109, 107, 110, 78, 139, 120, 40, 57, 73, 60, 65, 66, 14, 29, 31, 26, 22, 25, 8, 8, 10, 11, 16, 7],
                [131, 114, 139, 110, 125, 123, 104, 105, 118, 83, 95, 90, 13, 19, 18, 15, 17, 12, 10, 12, 7, 7, 7, 7],
                [114, 164, 154, 141, 129, 118, 112, 122, 119, 160, 184, 175, 30, 36, 34, 17, 33, 33, 19, 17, 8, 8, 17, 18],
                [119, 130, 132, 135, 124, 140, 87, 107, 109, 132, 134, 100, 22, 14, 22, 19, 23, 17, 10, 4, 8, 9, 6, 4],
                [146, 143, 127, 148, 123, 112, 81, 102, 82, 90, 106, 101, 35, 35, 28, 31, 34, 28, 9, 9, 8, 9, 11, 9]]
x = 0
for i in range(0, len(ground_truth)):
    print(f"Batch {i+1} number of samples:", len(ground_truth[i])) #total number -> 790
    x += len(ground_truth[i])
print("The total number of samples:", x)

# Mask sample 27

#### Partially segmented colonies with more accurate count (Part I - Segmentation)

In [80]:
# Contour objects <- https://docs.opencv.org/4.x/d4/d73/tutorial_py_contours_begin.html
# Count objects <- https://learnopencv.com/tag/cv2-moments/
# The colonies were partially segmented

numbers = re.compile(r'(\d+)')
def numericalSort(value):
    parts = numbers.split(value)
    parts[1::2] = map(int, parts[1::2])
    return parts

path_mask = '..\Data\HCT116\Masks_Binary\\'
path_overlay = '..\Data\HCT116\Masked_Binary\\'
mask_dir = sorted(os.listdir(path_mask), key=numericalSort)
overlay_dir = sorted(os.listdir(path_overlay), key=numericalSort)
nb_masks = len(os.listdir(path_mask))

for i, j in zip(mask_dir, overlay_dir):
    filename = i
    filename_overlay = j
    mask = cv.imread(path_mask + '\\' + filename)
    mask_uint8 = cv.cvtColor(cv.imread(path_mask + '\\' + filename).astype(np.uint8), cv.COLOR_RGB2GRAY)
    overlayed = cv.imread(path_overlay + '\\' + filename_overlay)
    overlay = overlayed.copy()
    overlay = cv.bitwise_not(cv.cvtColor(overlay, cv.COLOR_BGR2GRAY))
    
    thresh = cv.bitwise_and(overlay, mask_uint8)

    # cv.imshow("img", overlayed)
    # cv.waitKey(0)
    threshold_sureBack = cv.threshold(thresh, 115, 255, cv.THRESH_BINARY)[1] #110
    threshold_sureFront1 = cv.threshold(thresh, 239, 255, cv.THRESH_BINARY)[1] #240
    threshold_sureFront2 = cv.threshold(thresh, 7, 255, cv.THRESH_BINARY)[1] #7

    kernel = np.ones((3,3), np.uint8)
    threshold_sureFront2 = cv.erode(threshold_sureFront2, kernel, iterations=8) # use RGB
    threshold_sureFront2 = cv.dilate(threshold_sureFront2, kernel, iterations=2) # use RGB
    # cv.imshow("sureFront2", threshold_sureFront2)
    # cv.waitKey(0)

    # Remove small objects from the mask
    nb_blobs, mask_separated_blobs, stats, _ = cv.connectedComponentsWithStats(threshold_sureFront2)
    sizes = stats[:, cv.CC_STAT_AREA]
    sureFront2_processed = np.zeros_like(mask_separated_blobs)
    for i in range(1, nb_blobs):
        if sizes[i] <= 7000 and sizes[i] >= 200: #exclude big objects
            sureFront2_processed[mask_separated_blobs == i] = 255
    sureFront2_processed = img_as_ubyte(sureFront2_processed)
    

    # cv.imshow("img", sureFront2_processed)
    # cv.waitKey(0)
    # cv.imshow("img", threshold_sureBack)
    # cv.waitKey(0)

    # Noise removal
    kernel = np.ones((3,3), np.uint8)
    sureBack = cv.dilate(threshold_sureBack, kernel, iterations=5) #5
    threshold_sureFront = cv.erode(threshold_sureFront1, kernel, iterations=3) # use RGB
    # opening = cv.morphologyEx(threshold_sureBack, cv.MORPH_OPEN, kernel, iterations=5)
    # opening = cv.morphologyEx(threshold_sureFront, cv.MORPH_OPEN, kernel, iterations=5)
    threshold_sureFront = img_as_ubyte(morphology.remove_small_holes(threshold_sureFront, area_threshold=2000))
    threshold_sureFront = cv.medianBlur(threshold_sureFront, 15)



    destination2 = '..\Data\\' + 'threshold_sureFront.jpg'
    cv.imwrite(destination2, threshold_sureFront)

    # Further process non-solid objects
    contours, _ = cv.findContours(threshold_sureFront, cv.RETR_TREE, cv.CHAIN_APPROX_SIMPLE) #detect all objects in thresh1
    contours_solid=[]
    contours_non_solid = []
    for obj in contours:
        area = cv.contourArea(obj)
        hull = cv.convexHull(obj)
        hull_area = cv.contourArea(hull) # convex hull area
        solidity = float(area)/hull_area
        if(solidity<0.94 and area>500):
            contours_non_solid.append(obj)
        else:
            contours_solid.append(obj)
    contours_non_solid = tuple(contours_non_solid)
    contours_solid = tuple(contours_solid)

    # Create new masks for process of non-solid objects
    new_mask = np.zeros(overlay.shape, dtype=np.uint8)
    threshold_sureFront = np.zeros(overlay.shape, dtype=np.uint8)
    cv.drawContours(new_mask, contours_non_solid, -1, color=(255, 255, 255), thickness=cv.FILLED)
    cv.drawContours(threshold_sureFront, contours_solid, -1, color=(255, 255, 255), thickness=cv.FILLED) #split the mask further
    destination2 = '..\Data\\' + 'newmask.jpg'
    cv.imwrite(destination2, new_mask)

    dist_transform = cv.distanceTransform(new_mask,cv.DIST_L2,5)
    _, new_mask = cv.threshold(dist_transform,0.15*dist_transform.max(),255,0)
    kernel1 = cv.getStructuringElement(cv.MORPH_CROSS,(7,7))
    kernel2 = cv.getStructuringElement(cv.MORPH_ELLIPSE,(5,5))
    # new_mask = cv.dilate(new_mask, kernel2, iterations=2)
    new_mask = cv.erode(new_mask, kernel2, iterations=9)
    new_mask = cv.morphologyEx(new_mask, cv.MORPH_CLOSE, kernel, iterations=3)
    threshold_sureFront = cv.dilate(threshold_sureFront, kernel2, iterations=2)

    destination2 = '..\Data\\' + 'newmask_transformed.jpg'
    cv.imwrite(destination2, new_mask)

    print(type(threshold_sureFront))
    print(type(new_mask))

    raw_sureFront = cv.bitwise_or(np.array(threshold_sureFront, dtype=np.uint8), np.array(new_mask, dtype=np.uint8))
    destination2 = '..\Data\\' + 'raw_sureFront.jpg'
    cv.imwrite(destination2, raw_sureFront)


    # destination2 = '..\Data\\' + 'sureFront1.jpg'
    # cv.imwrite(destination2, threshold_sureFront)
    # destination2 = '..\Data\\' + 'sureBack.jpg'
    # cv.imwrite(destination2, sureBack)

    sureFront = cv.bitwise_or(raw_sureFront, sureFront2_processed)

    unsure = cv.subtract(sureBack, sureFront)

    # Marker labelling
    ret, markers = cv.connectedComponents(sureFront)
    markers += 1
    markers[unsure==255] = 0

    # Watershed
    markers = cv.watershed(mask, markers)
    overlayed[markers==-1] = [0, 255, 0]

    markers = markers.astype(np.uint8)
    ret, thresh2 = cv.threshold(markers, 0, 255, cv.THRESH_BINARY|cv.THRESH_OTSU)
    contours, hierarchy = cv.findContours(thresh2, cv.RETR_LIST, cv.CHAIN_APPROX_SIMPLE)

    print(filename)
    cv.cvtColor(cv.drawContours(mask, contours, -1, (0, 0, 0), 10), cv.COLOR_RGB2GRAY)
    # experimental.append(counter)


    destination2 = '..\Data\\' + 'mask.jpg'
    cv.imwrite(destination2, mask)


    cv.imshow("img", thresh2)
    cv.waitKey(0)

    
    # destination2 = '..\Data\HCT116\Test_new\\' + filename
    # cv.imwrite(destination2, mask)

c:\Users\monaw\Anaconda3\envs\venv\lib\site-packages\skimage\util\dtype.py:576: UserWarning: Downcasting int32 to uint8 without scaling because max value 255 fits in uint8
  return _convert(image, np.uint8, force_copy)
C:\Users\monaw\AppData\Local\Temp\ipykernel_1284\2475362022.py:61: UserWarning: Any labeled images will be returned as a boolean array. Did you mean to use a boolean array?
  threshold_sureFront = img_as_ubyte(morphology.remove_small_holes(threshold_sureFront, area_threshold=2000))


<class 'numpy.ndarray'>
<class 'numpy.ndarray'>
Mask_Sample_1-1.jpg


KeyboardInterrupt: 

#### Partially segmented colonies with accurate count (Part II - Colony Count)

In [52]:
# Count the colonies in the segmented images
import re

ground_truth = [[113, 90, 99, 87, 96, 93, 92, 95, 96, 76, 60, 63, 10, 11, 10, 1, 1, 1],
                [145, 175, 160, 167, 156, 120, 395, 446, 436, 107, 101, 92, 43, 28, 45, 4, 3, 6],
                [103, 112, 72, 74, 76, 100, 62, 74, 79, 71, 83, 79, 8, 11, 10, 3, 2, 12],
                [170, 176, 160, 139, 137, 156, 101, 98, 93, 96, 102, 87, 27, 29, 20, 7, 4, 4],
                [165, 159, 129, 149, 164, 160, 90, 89, 103, 116, 127, 128, 18, 18, 16, 3, 8, 9],
                [139, 140, 123, 117, 106, 121, 89, 100, 79, 91, 100, 102, 14, 22, 9, 7, 10, 7],
                [160, 153, 144, 140, 140, 151, 149, 143, 146, 166, 168, 169, 25, 20, 20, 14, 11, 9],
                [157, 160, 117, 138, 140, 155, 144, 118, 117, 138, 125, 108, 27, 29, 25, 6, 11, 10],
                [169, 156, 173, 217, 171, 170, 123, 125, 109, 166, 143, 173, 32, 19, 21, 8, 6, 4],
                [153, 156, 135, 157, 138, 163, 94, 92, 72, 119, 117, 92, 31, 31, 21, 4, 9, 8],
                [147, 158, 143, 156, 149, 119, 118, 143, 121, 168, 175, 144, 28, 27, 18, 10, 8, 13],
                [137, 115, 115, 126, 122, 136, 93, 105, 114, 137, 119, 142, 28, 29, 25, 8, 12, 7],
                [228, 240, 294, 259, 243, 239, 1, 1, 0],
                [114, 98, 115, 104, 98, 121, 99, 75, 75, 59, 76, 62, 9, 9, 14, 4, 1, 0],
                [220, 202, 213, 176, 209, 201, 112, 127, 101, 65, 62, 64, 18, 23, 31, 5, 3, 5],
                [145, 152, 167, 164, 167, 166, 114, 82, 74, 59, 12, 10, 13, 1, 3, 4],
                [61, 59, 62, 60, 67, 57, 30, 19, 36, 33, 45, 32, 6, 7, 16, 16, 16, 7, 15, 15, 17, 33, 28, 28],
                [103, 95, 87, 87, 83, 90, 44, 48, 51, 45, 63, 55, 13, 19, 17, 19, 20, 21, 6, 7, 2, 4, 2, 4],
                [74, 72, 93, 65, 76, 87, 60, 55, 49, 59, 49, 66, 16, 11, 9, 15, 12, 14, 4, 2, 5, 7, 4, 8],
                [8, 11, 13, 6, 10, 5, 46, 55, 53, 69, 67, 52, 8, 11, 14, 6, 19, 12, 12, 18, 11, 14, 15, 11],
                [70, 90, 80, 96, 93, 76, 68, 63, 65, 78, 75, 59, 10, 6, 10, 8, 16, 9, 3, 2, 4, 6, 5, 4],
                [43, 48, 46, 34, 41, 41, 12, 16, 8, 25, 24, 20, 3, 7, 12, 6, 7, 5, 4, 2, 2, 1, 2, 0],
                [133, 130, 99, 105, 117, 110, 34, 49, 55, 48, 55, 65, 17, 22, 14, 14, 16, 13, 2, 2, 3, 8, 1, 6],
                [29, 40, 5, 12, 9, 6],
                [118, 102, 108, 101, 112, 113, 46, 45, 35, 47, 54, 45, 11, 16, 16, 12, 20, 17, 3, 2, 1, 4, 3, 1],
                [144, 117, 147, 149, 125, 122, 56, 73, 52, 40, 55, 51, 21, 29, 30, 23, 19, 32, 9, 5, 3, 8, 3, 5],
                [131, 148, 160, 155, 158, 132, 96, 110, 79, 116, 123, 108, 46, 33, 46, 40, 45, 42, 42, 26, 40, 42, 30, 36], #27
                [157, 144, 143, 142, 159, 163, 78, 78, 70, 149, 153, 124, 53, 43, 44, 49, 45, 45, 28, 26, 22, 24, 20, 20],
                [134, 128, 108, 121, 111, 105, 94, 85, 84, 110, 126, 116, 53, 48, 49, 51, 51, 40, 41, 46, 36, 38, 42, 37],
                [159, 134, 163, 165, 179, 147, 94, 95, 92, 158, 149, 158, 41, 39, 39, 38, 45, 47, 36, 30, 32, 34, 36, 25],
                [181, 136, 161, 158, 155, 130, 70, 74, 72, 128, 140, 125, 51, 42, 46, 52, 57, 48, 18, 25, 27, 20, 23, 19],
                [137, 141, 137, 147, 158, 124, 82, 75, 80, 166, 143, 130, 28, 34, 39, 47, 39, 46, 17, 25, 21, 19, 22, 20],
                [146, 146, 164, 125, 150, 160, 111, 110, 98, 134, 120, 129, 64, 46, 52, 60, 43, 55, 36, 34, 42, 31, 28, 36],
                [109, 107, 110, 78, 139, 120, 40, 57, 73, 60, 65, 66, 14, 29, 31, 26, 22, 25, 8, 8, 10, 11, 16, 7],
                [131, 114, 139, 110, 125, 123, 104, 105, 118, 83, 95, 90, 13, 19, 18, 15, 17, 12, 10, 12, 7, 7, 7, 7],
                [114, 164, 154, 141, 129, 118, 112, 122, 119, 160, 184, 175, 30, 36, 34, 17, 33, 33, 19, 17, 8, 8, 17, 18],
                [119, 130, 132, 135, 124, 140, 87, 107, 109, 132, 134, 100, 22, 14, 22, 19, 23, 17, 10, 4, 8, 9, 6, 4],
                [146, 143, 127, 148, 123, 112, 81, 102, 82, 90, 106, 101, 35, 35, 28, 31, 34, 28, 9, 9, 8, 9, 11, 9]]

def accuracy_calc(ground_truth, experimental, batch):
    percent_diff = []
    for i in range(len(ground_truth[batch-1])):
        if ground_truth[batch-1][i] == 0:
            percent_diff.append(0) #may change later
        else:
            percent_diff.append(abs((experimental[batch-1][i] - ground_truth[batch-1][i]) / ground_truth[batch-1][i]))
    avg_percent_diff = sum(percent_diff) / len(percent_diff)
    accuracy = (1 - avg_percent_diff) * 100
    return accuracy

experimental = []
total_accuracy = []

# For counting in numerical order
numbers = re.compile(r'(\d+)')
def numericalSort(value): #https://stackoverflow.com/questions/12093940/reading-files-in-a-particular-order-in-python
    parts = numbers.split(value)
    parts[1::2] = map(int, parts[1::2])
    return parts

path = '..\Data\HCT116\Test_new\\'
dir = sorted(os.listdir(path), key=numericalSort)

for batch in range(1, len(ground_truth) + 1):
    temp = []
    print(f"number of samples in batch {batch} is:", len(ground_truth[batch-1]))
    for sample in range(1, len(ground_truth[batch-1]) + 1):
        filename = "Mask_Sample_" + str(batch) + "-" + str(sample) + ".jpg"
        original = cv.imread(path + filename)
        img = cv.cvtColor(original, cv.COLOR_BGR2GRAY)
        img = img.astype(np.uint8)
        ret, thresh = cv.threshold(img, 15, 255, cv.THRESH_BINARY|cv.THRESH_OTSU)
        contours, hierarchy = cv.findContours(thresh, cv.RETR_LIST, cv.CHAIN_APPROX_SIMPLE)
    
        counter = 0
        for contour in contours:
            # Compute moments of the contour
            M = cv.moments(contour)
            if M['m00'] != 0:
                cx = int(M['m10'] / M['m00'])
                cy = int(M['m01'] / M['m00'])
                # Draw circle at center of mass
                cv.circle(original, (cx, cy), 5, (0, 255, 0), -1)
                counter += 1

        print(f"{filename} has {counter} colonies")
        cv.drawContours(original, contours, -1, (0, 255, 0), 5)
        temp.append(counter)

        # cv.imshow("img", original)
        # cv.waitKey(0)
        destination = '..\Data\HCT116\Test_Count_new\\' + filename
        cv.imwrite(destination, original)
    
    experimental.append(temp)
    accuracy = accuracy_calc(ground_truth=ground_truth, experimental=experimental, batch=batch)
    total_accuracy.append(accuracy)
    print(f"The accuracy for batch {batch} is {accuracy}")

avg_accuracy = sum(total_accuracy) / len(total_accuracy)
print(f"The averaged accuracy for the entire dataset is {avg_accuracy}")


FileNotFoundError: [WinError 3] The system cannot find the path specified: '..\\Data\\HCT116\\Test_new\\'